# Atlassian MCP Server Connection Test

This notebook tests connectivity to Atlassian's Model Context Protocol (MCP) server for Confluence.

## Overview

The Atlassian MCP server provides a standardized interface to interact with Confluence (and Jira) using the Model Context Protocol. This allows LLM applications to:
- Search Confluence pages
- Read page content
- Create and update pages
- Execute natural language queries

**Server Endpoint:** `https://mcp.atlassian.com/v1/mcp`

**Authentication:**
- OAuth 2.1 (recommended) - browser-based authorization
- API Token - optional, must be enabled by org admin

## References
- [Atlassian MCP Server Docs](https://support.atlassian.com/atlassian-rovo-mcp-server/docs/getting-started-with-the-atlassian-remote-mcp-server/)
- [MCP Protocol Specification](https://modelcontextprotocol.io/docs/concepts/transports)

## 1. Setup and Dependencies

In [1]:
import sys
sys.path.append('..')

import os
import json
import asyncio
from datetime import datetime
from typing import Optional, Dict, Any, List
from dataclasses import dataclass
from loguru import logger

# Configure logging for detailed output
logger.remove()
logger.add(sys.stderr, level="DEBUG", format="<green>{time:HH:mm:ss}</green> | <level>{level: <8}</level> | {message}")

print("Basic imports successful")

Basic imports successful


In [2]:
# Check for MCP SDK
try:
    import mcp
    from mcp import ClientSession
    print(f"MCP SDK version: {mcp.__version__ if hasattr(mcp, '__version__') else 'unknown'}")
    MCP_SDK_AVAILABLE = True
except ImportError as e:
    print(f"MCP SDK not installed: {e}")
    print("\nTo install, run:")
    print('  pip install "mcp[cli]"')
    print("  # or with uv:")
    print('  uv add "mcp[cli]"')
    MCP_SDK_AVAILABLE = False

# Check for httpx (for manual HTTP testing)
try:
    import httpx
    print(f"httpx available: {httpx.__version__}")
    HTTPX_AVAILABLE = True
except ImportError:
    print("httpx not installed (optional, for manual testing)")
    print("  pip install httpx")
    HTTPX_AVAILABLE = False

# Check for httpx-sse (for SSE support)
try:
    import httpx_sse
    print("httpx-sse available")
    HTTPX_SSE_AVAILABLE = True
except ImportError:
    print("httpx-sse not installed (optional, for SSE support)")
    print("  pip install httpx-sse")
    HTTPX_SSE_AVAILABLE = False

MCP SDK version: unknown
httpx available: 0.28.1
httpx-sse available


## 2. Configuration

Configure credentials for the Atlassian MCP server.

**Options:**
1. **API Token** - Set `ATLASSIAN_API_TOKEN` environment variable (requires org admin to enable)
2. **OAuth** - Will prompt for browser-based authentication
3. **Site-specific** - Set `ATLASSIAN_SITE_URL` for your Confluence instance

In [3]:
from dotenv import load_dotenv
load_dotenv()

@dataclass
class AtlassianMCPConfig:
    """Configuration for Atlassian MCP server connection."""
    
    # MCP server endpoint
    mcp_endpoint: str = "https://mcp.atlassian.com/v1/mcp"
    
    # Legacy SSE endpoint (deprecated after June 30, 2026)
    sse_endpoint: str = "https://mcp.atlassian.com/v1/sse"
    
    # Your Atlassian site URL (e.g., https://yoursite.atlassian.net)
    site_url: Optional[str] = None
    
    # API token (if using token auth instead of OAuth)
    api_token: Optional[str] = None
    
    # User email (for API token auth)
    user_email: Optional[str] = None
    
    # Protocol version
    protocol_version: str = "2025-06-18"
    
    @classmethod
    def from_env(cls) -> "AtlassianMCPConfig":
        """Load configuration from environment variables."""
        return cls(
            site_url=os.getenv("ATLASSIAN_SITE_URL") or os.getenv("CONFLUENCE_URL"),
            api_token=os.getenv("ATLASSIAN_API_TOKEN") or os.getenv("CONFLUENCE_API_TOKEN"),
            user_email=os.getenv("ATLASSIAN_USER_EMAIL") or os.getenv("CONFLUENCE_USERNAME"),
        )
    
    def validate(self) -> Dict[str, Any]:
        """Validate configuration and return status."""
        status = {
            "mcp_endpoint": self.mcp_endpoint,
            "site_url_configured": bool(self.site_url),
            "api_token_configured": bool(self.api_token),
            "user_email_configured": bool(self.user_email),
            "auth_method": "api_token" if self.api_token else "oauth",
            "issues": [],
        }
        
        if not self.site_url:
            status["issues"].append("ATLASSIAN_SITE_URL not set - you may need to configure your site")
        
        if self.api_token and not self.user_email:
            status["issues"].append("API token provided but ATLASSIAN_USER_EMAIL not set")
        
        return status

# Load configuration
config = AtlassianMCPConfig.from_env()
validation = config.validate()

print("=" * 60)
print("ATLASSIAN MCP CONFIGURATION")
print("=" * 60)
print(f"MCP Endpoint: {validation['mcp_endpoint']}")
print(f"Site URL: {config.site_url or 'NOT SET'}")
print(f"Auth Method: {validation['auth_method']}")
print(f"API Token: {'*' * 8 + config.api_token[-4:] if config.api_token else 'NOT SET'}")
print(f"User Email: {config.user_email or 'NOT SET'}")
print()
if validation["issues"]:
    print("ISSUES:")
    for issue in validation["issues"]:
        print(f"  - {issue}")
else:
    print("Configuration looks good!")

ATLASSIAN MCP CONFIGURATION
MCP Endpoint: https://mcp.atlassian.com/v1/mcp
Site URL: https://confluence.abbvienet.com/
Auth Method: api_token
API Token: ********qt4w
User Email: garrick.bruening@abbvie.com

Configuration looks good!


## 3. Low-Level HTTP Connection Test

Before using the MCP SDK, let's test basic HTTP connectivity to the Atlassian MCP endpoint.
This helps diagnose network issues, authentication problems, and endpoint availability.

In [4]:
import requests
from urllib.parse import urlparse
import warnings

# Suppress InsecureRequestWarning for testing
from urllib3.exceptions import InsecureRequestWarning
warnings.filterwarnings('ignore', category=InsecureRequestWarning)

def test_endpoint_connectivity(url: str, timeout: int = 10) -> Dict[str, Any]:
    """Test basic HTTP connectivity to an endpoint.
    
    Returns detailed diagnostic information.
    """
    result = {
        "url": url,
        "reachable": False,
        "status_code": None,
        "response_time_ms": None,
        "headers": {},
        "error": None,
        "dns_resolved": False,
        "tls_version": None,
    }
    
    parsed = urlparse(url)
    
    # Test DNS resolution
    import socket
    try:
        socket.gethostbyname(parsed.netloc)
        result["dns_resolved"] = True
    except socket.gaierror as e:
        result["error"] = f"DNS resolution failed: {e}"
        return result
    
    # Test HTTP connection
    import time
    start = time.time()
    
    try:
        # Try OPTIONS first (less invasive)
        # Note: verify=False bypasses SSL cert verification for corporate proxy environments
        response = requests.options(url, timeout=timeout, verify=False)
        result["response_time_ms"] = round((time.time() - start) * 1000)
        result["reachable"] = True
        result["status_code"] = response.status_code
        result["headers"] = dict(response.headers)
        
    except requests.exceptions.SSLError as e:
        result["error"] = f"SSL/TLS error: {e}"
    except requests.exceptions.ConnectionError as e:
        result["error"] = f"Connection error: {e}"
    except requests.exceptions.Timeout as e:
        result["error"] = f"Timeout after {timeout}s: {e}"
    except requests.exceptions.RequestException as e:
        result["error"] = f"Request error: {e}"
    
    return result

print("Testing connectivity to Atlassian MCP endpoint...")
print()

connectivity = test_endpoint_connectivity(config.mcp_endpoint)

print("=" * 60)
print("CONNECTIVITY TEST RESULTS")
print("=" * 60)
print(f"URL: {connectivity['url']}")
print(f"DNS Resolved: {connectivity['dns_resolved']}")
print(f"Reachable: {connectivity['reachable']}")
print(f"Status Code: {connectivity['status_code']}")
print(f"Response Time: {connectivity['response_time_ms']}ms")

if connectivity['error']:
    print(f"\nERROR: {connectivity['error']}")
    print("\nTroubleshooting:")
    if "DNS" in connectivity['error']:
        print("  - Check your network connection")
        print("  - Verify you can reach atlassian.com")
    elif "SSL" in connectivity['error']:
        print("  - Check if your organization uses a proxy")
        print("  - Verify SSL certificates are up to date")
    elif "Connection" in connectivity['error']:
        print("  - Check firewall settings")
        print("  - Verify VPN connection if required")
else:
    print("\nConnection successful!")
    if connectivity['headers']:
        print("\nResponse Headers:")
        for k, v in list(connectivity['headers'].items())[:5]:
            print(f"  {k}: {v[:50]}..." if len(str(v)) > 50 else f"  {k}: {v}")

Testing connectivity to Atlassian MCP endpoint...

CONNECTIVITY TEST RESULTS
URL: https://mcp.atlassian.com/v1/mcp
DNS Resolved: True
Reachable: True
Status Code: 204
Response Time: 297ms

Connection successful!

Response Headers:
  Date: Sun, 08 Mar 2026 15:51:45 GMT
  Cf-Ray: 9d9300cd6fd78ce4-PDX
  Vary: Accept-Encoding
  Server: AtlassianEdge
  Ge-Edge-Trusted-Cloudflare-Proxy: bWNwLWNsb3VkZmxhcmUK


## 4. MCP Protocol Test (Manual)

Test the MCP protocol handshake manually using HTTP requests.
This helps debug protocol-level issues before using the SDK.

In [5]:
def build_mcp_headers(config: AtlassianMCPConfig, session_id: Optional[str] = None) -> Dict[str, str]:
    """Build headers for MCP requests."""
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/event-stream",
        "MCP-Protocol-Version": config.protocol_version,
    }
    
    # Add session ID if we have one
    if session_id:
        headers["Mcp-Session-Id"] = session_id
    
    # Add API token auth if configured
    if config.api_token and config.user_email:
        import base64
        credentials = f"{config.user_email}:{config.api_token}"
        encoded = base64.b64encode(credentials.encode()).decode()
        headers["Authorization"] = f"Basic {encoded}"
    
    return headers

def create_initialize_request(request_id: int = 1) -> Dict[str, Any]:
    """Create an MCP initialize request."""
    return {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": "initialize",
        "params": {
            "protocolVersion": "2025-06-18",
            "capabilities": {
                "tools": {},
                "resources": {},
                "prompts": {},
            },
            "clientInfo": {
                "name": "confluence-rag-test",
                "version": "1.0.0"
            }
        }
    }

def send_mcp_request(
    endpoint: str,
    request: Dict[str, Any],
    headers: Dict[str, str],
    timeout: int = 30
) -> Dict[str, Any]:
    """Send an MCP request and return the response with diagnostics."""
    result = {
        "success": False,
        "request": request,
        "status_code": None,
        "response_headers": {},
        "response_body": None,
        "session_id": None,
        "error": None,
        "is_sse": False,
    }
    
    try:
        response = requests.post(
            endpoint,
            json=request,
            headers=headers,
            timeout=timeout,
            stream=True,  # Enable streaming for SSE
        )
        
        result["status_code"] = response.status_code
        result["response_headers"] = dict(response.headers)
        
        # Check for session ID
        if "Mcp-Session-Id" in response.headers:
            result["session_id"] = response.headers["Mcp-Session-Id"]
        
        # Check content type
        content_type = response.headers.get("Content-Type", "")
        
        if "text/event-stream" in content_type:
            # SSE response - read events
            result["is_sse"] = True
            events = []
            for line in response.iter_lines(decode_unicode=True):
                if line:
                    events.append(line)
                    # Stop after getting a reasonable amount
                    if len(events) > 20:
                        break
            result["response_body"] = events
            result["success"] = response.status_code < 400
            
        elif "application/json" in content_type:
            # JSON response
            result["response_body"] = response.json()
            result["success"] = response.status_code < 400
            
        else:
            # Unknown content type
            result["response_body"] = response.text[:1000]
            result["success"] = response.status_code < 400
            
    except requests.exceptions.RequestException as e:
        result["error"] = str(e)
    
    return result

print("Testing MCP protocol handshake...")
print()

Testing MCP protocol handshake...



In [6]:
# Send initialize request
headers = build_mcp_headers(config)
init_request = create_initialize_request()

print("Request:")
print(json.dumps(init_request, indent=2))
print()
print("Headers:")
for k, v in headers.items():
    if k == "Authorization":
        print(f"  {k}: [REDACTED]")
    else:
        print(f"  {k}: {v}")
print()

# Note: Adding verify=False to send_mcp_request for corporate proxy environments
def send_mcp_request_with_ssl_bypass(
    endpoint: str,
    request: Dict[str, Any],
    headers: Dict[str, str],
    timeout: int = 30
) -> Dict[str, Any]:
    """Send an MCP request with SSL verification disabled."""
    result = {
        "success": False,
        "request": request,
        "status_code": None,
        "response_headers": {},
        "response_body": None,
        "session_id": None,
        "error": None,
        "is_sse": False,
    }
    
    try:
        response = requests.post(
            endpoint,
            json=request,
            headers=headers,
            timeout=timeout,
            stream=True,
            verify=False,  # Bypass SSL verification for corporate proxy
        )
        
        result["status_code"] = response.status_code
        result["response_headers"] = dict(response.headers)
        
        # Check for session ID
        if "Mcp-Session-Id" in response.headers:
            result["session_id"] = response.headers["Mcp-Session-Id"]
        
        # Check content type
        content_type = response.headers.get("Content-Type", "")
        
        if "text/event-stream" in content_type:
            result["is_sse"] = True
            events = []
            for line in response.iter_lines(decode_unicode=True):
                if line:
                    events.append(line)
                    if len(events) > 20:
                        break
            result["response_body"] = events
            result["success"] = response.status_code < 400
            
        elif "application/json" in content_type:
            result["response_body"] = response.json()
            result["success"] = response.status_code < 400
            
        else:
            result["response_body"] = response.text[:1000]
            result["success"] = response.status_code < 400
            
    except requests.exceptions.RequestException as e:
        result["error"] = str(e)
    
    return result

result = send_mcp_request_with_ssl_bypass(config.mcp_endpoint, init_request, headers)

print("=" * 60)
print("MCP INITIALIZE RESPONSE")
print("=" * 60)
print(f"Success: {result['success']}")
print(f"Status Code: {result['status_code']}")
print(f"Is SSE: {result['is_sse']}")
print(f"Session ID: {result['session_id']}")

if result['error']:
    print(f"\nERROR: {result['error']}")
elif result['response_body']:
    print("\nResponse Body:")
    if isinstance(result['response_body'], (dict, list)):
        print(json.dumps(result['response_body'], indent=2)[:2000])
    else:
        print(result['response_body'][:2000])

# Interpret common status codes
if result['status_code']:
    print("\nStatus Code Interpretation:")
    interpretations = {
        200: "Success - Server accepted the request",
        401: "Unauthorized - Authentication required (try OAuth or check API token)",
        403: "Forbidden - Valid auth but insufficient permissions",
        404: "Not Found - Endpoint may have changed or site not configured",
        405: "Method Not Allowed - Server doesn't accept POST (try legacy SSE endpoint)",
        429: "Rate Limited - Too many requests",
        500: "Server Error - Atlassian service issue",
        502: "Bad Gateway - Proxy or load balancer issue",
        503: "Service Unavailable - Server temporarily down",
    }
    print(f"  {interpretations.get(result['status_code'], 'Unknown status code')}")

Request:
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "initialize",
  "params": {
    "protocolVersion": "2025-06-18",
    "capabilities": {
      "tools": {},
      "resources": {},
      "prompts": {}
    },
    "clientInfo": {
      "name": "confluence-rag-test",
      "version": "1.0.0"
    }
  }
}

Headers:
  Content-Type: application/json
  Accept: application/json, text/event-stream
  MCP-Protocol-Version: 2025-06-18
  Authorization: [REDACTED]

MCP INITIALIZE RESPONSE
Success: True
Status Code: 200
Is SSE: True
Session ID: r13-060b9c1a-9070-4e7c-a719-a9042444a143

Response Body:
[
  "event: message",
  "data: {\"result\":{\"protocolVersion\":\"2025-06-18\",\"capabilities\":{\"logging\":{},\"tools\":{},\"resources\":{}},\"serverInfo\":{\"name\":\"atlassian-mcp-server\",\"version\":\"1.0.0\"}},\"jsonrpc\":\"2.0\",\"id\":1}"
]

Status Code Interpretation:
  Success - Server accepted the request


## 5. MCP SDK Connection Test

If the MCP SDK is installed, test the connection using the official client.
This provides a higher-level interface and handles the protocol details.

In [7]:
if not MCP_SDK_AVAILABLE:
    print("MCP SDK not available. Skipping SDK tests.")
    print("Install with: pip install 'mcp[cli]'")
else:
    print("MCP SDK is available. Proceeding with SDK tests...")

MCP SDK is available. Proceeding with SDK tests...


In [8]:
# MCP SDK Connection Test
if MCP_SDK_AVAILABLE:
    from mcp import ClientSession
    
    # Try to import the streamable HTTP client
    try:
        from mcp.client.streamable_http import streamable_http_client
        STREAMABLE_HTTP_AVAILABLE = True
        print("Streamable HTTP client available")
    except ImportError:
        STREAMABLE_HTTP_AVAILABLE = False
        print("Streamable HTTP client not available")
    
    # Try to import SSE client
    try:
        from mcp.client.sse import sse_client
        SSE_CLIENT_AVAILABLE = True
        print("SSE client available")
    except ImportError:
        SSE_CLIENT_AVAILABLE = False
        print("SSE client not available")

Streamable HTTP client available
SSE client available


In [9]:
async def test_mcp_sdk_connection(endpoint: str, headers: Optional[Dict[str, str]] = None):
    """Test MCP connection using the SDK.
    
    Returns detailed connection status and available capabilities.
    
    Note: Uses httpx.AsyncClient with verify=False to bypass SSL verification
    in corporate proxy environments.
    """
    result = {
        "connected": False,
        "initialized": False,
        "server_info": None,
        "capabilities": None,
        "tools": [],
        "resources": [],
        "prompts": [],
        "error": None,
        "error_details": None,
    }
    
    if not MCP_SDK_AVAILABLE:
        result["error"] = "MCP SDK not installed"
        return result
    
    if not STREAMABLE_HTTP_AVAILABLE:
        result["error"] = "Streamable HTTP client not available in MCP SDK"
        return result
    
    try:
        # Build custom headers if needed
        custom_headers = headers or {}
        
        # Create httpx.AsyncClient with headers and SSL bypass
        # This is the correct pattern - streamable_http_client doesn't accept headers directly
        async with httpx.AsyncClient(headers=custom_headers, verify=False) as http_client:
            async with streamable_http_client(endpoint, http_client=http_client) as (read, write, _):
                result["connected"] = True
                logger.info("Connected to MCP server")
                
                async with ClientSession(read, write) as session:
                    # Initialize the session
                    init_result = await session.initialize()
                    result["initialized"] = True
                    result["server_info"] = getattr(init_result, 'serverInfo', None)
                    result["capabilities"] = getattr(init_result, 'capabilities', None)
                    
                    logger.info(f"Session initialized. Server: {result['server_info']}")
                    
                    # List available tools
                    try:
                        tools_result = await session.list_tools()
                        result["tools"] = [{
                            "name": t.name,
                            "description": getattr(t, 'description', None),
                        } for t in tools_result.tools]
                        logger.info(f"Found {len(result['tools'])} tools")
                    except Exception as e:
                        logger.warning(f"Could not list tools: {e}")
                    
                    # List available resources
                    try:
                        resources_result = await session.list_resources()
                        result["resources"] = [{
                            "uri": r.uri,
                            "name": getattr(r, 'name', None),
                        } for r in resources_result.resources]
                        logger.info(f"Found {len(result['resources'])} resources")
                    except Exception as e:
                        logger.warning(f"Could not list resources: {e}")
                    
                    # List available prompts
                    try:
                        prompts_result = await session.list_prompts()
                        result["prompts"] = [{
                            "name": p.name,
                            "description": getattr(p, 'description', None),
                        } for p in prompts_result.prompts]
                        logger.info(f"Found {len(result['prompts'])} prompts")
                    except Exception as e:
                        logger.warning(f"Could not list prompts: {e}")
                    
    except Exception as e:
        result["error"] = str(e)
        result["error_details"] = {
            "type": type(e).__name__,
            "message": str(e),
        }
        logger.error(f"Connection failed: {e}")
    
    return result

print("SDK connection test function defined (with SSL bypass and correct httpx.AsyncClient pattern).")

SDK connection test function defined (with SSL bypass and correct httpx.AsyncClient pattern).


In [10]:
# Run the SDK connection test
if MCP_SDK_AVAILABLE and STREAMABLE_HTTP_AVAILABLE:
    print("Testing MCP SDK connection...")
    print()
    
    # Build headers with auth if configured
    sdk_headers = {}
    if config.api_token and config.user_email:
        import base64
        credentials = f"{config.user_email}:{config.api_token}"
        encoded = base64.b64encode(credentials.encode()).decode()
        sdk_headers["Authorization"] = f"Basic {encoded}"
        print("Using API token authentication")
    else:
        print("No API token - OAuth may be required")
    
    # Run the async test
    sdk_result = await test_mcp_sdk_connection(config.mcp_endpoint, sdk_headers)
    
    print()
    print("=" * 60)
    print("MCP SDK CONNECTION RESULTS")
    print("=" * 60)
    print(f"Connected: {sdk_result['connected']}")
    print(f"Initialized: {sdk_result['initialized']}")
    
    if sdk_result['server_info']:
        print(f"Server Info: {sdk_result['server_info']}")
    
    if sdk_result['tools']:
        print(f"\nAvailable Tools ({len(sdk_result['tools'])})")
        for tool in sdk_result['tools'][:10]:  # Show first 10
            print(f"  - {tool['name']}: {tool.get('description', 'No description')[:60]}")
    
    if sdk_result['error']:
        print(f"\nERROR: {sdk_result['error']}")
        if sdk_result['error_details']:
            print(f"  Type: {sdk_result['error_details']['type']}")
else:
    print("Skipping SDK test - dependencies not available")

09:51:53 | INFO     | Connected to MCP server


Testing MCP SDK connection...

Using API token authentication


09:51:53 | INFO     | Session initialized. Server: name='atlassian-mcp-server' title=None version='1.0.0' websiteUrl=None icons=None
09:51:54 | WARNING  | Could not list tools: Method not found
09:51:54 | WARNING  | Could not list resources: Method not found
09:51:54 | WARNING  | Could not list prompts: Method not found



MCP SDK CONNECTION RESULTS
Connected: True
Initialized: True
Server Info: name='atlassian-mcp-server' title=None version='1.0.0' websiteUrl=None icons=None


## 6. Direct Tool Call Test (Manual HTTP)

The Atlassian MCP server doesn't support `tools/list` method (returns "Method not found").
However, we can try calling tools directly by name. Let's test with known Atlassian tool patterns.

In [11]:
# Test direct tool calls and method discovery via manual HTTP
# Since list_tools doesn't work, we'll probe the server to find what methods it supports

def send_mcp_method(
    endpoint: str,
    method: str,
    params: Dict[str, Any],
    headers: Dict[str, str],
    session_id: Optional[str] = None,
    request_id: int = 1,
) -> Dict[str, Any]:
    """Send any MCP method via manual HTTP."""
    
    # Add session ID to headers if provided
    if session_id:
        headers = {**headers, "Mcp-Session-Id": session_id}
    
    request = {
        "jsonrpc": "2.0",
        "id": request_id,
        "method": method,
        "params": params,
    }
    
    try:
        response = requests.post(
            endpoint,
            json=request,
            headers=headers,
            timeout=30,
            stream=True,
            verify=False,
        )
        
        content_type = response.headers.get("Content-Type", "")
        
        if "text/event-stream" in content_type:
            # Parse SSE response
            for line in response.iter_lines(decode_unicode=True):
                if line and line.startswith("data:"):
                    data = line[5:].strip()
                    try:
                        return {"success": True, "response": json.loads(data)}
                    except json.JSONDecodeError:
                        return {"success": False, "error": f"Invalid JSON: {data}"}
            return {"success": False, "error": "No data in SSE response"}
        elif "application/json" in content_type:
            return {"success": True, "response": response.json()}
        else:
            return {"success": False, "error": f"Unexpected content type: {content_type}", "body": response.text[:500]}
            
    except Exception as e:
        return {"success": False, "error": str(e)}

# First, establish a session
print("=" * 60)
print("ESTABLISHING MCP SESSION")
print("=" * 60)

headers = build_mcp_headers(config)
init_result = send_mcp_method(
    config.mcp_endpoint,
    "initialize",
    {
        "protocolVersion": "2025-06-18",
        "capabilities": {"tools": {}, "resources": {}, "prompts": {}},
        "clientInfo": {"name": "confluence-rag-manual", "version": "1.0.0"}
    },
    headers
)

if init_result["success"]:
    session_id = init_result["response"].get("id")
    print(f"Session established!")
    print(f"Server: {init_result['response'].get('result', {}).get('serverInfo', {})}")
else:
    print(f"Failed to initialize: {init_result.get('error')}")
    session_id = None

# Send initialized notification (required by MCP protocol)
if session_id:
    notify_result = send_mcp_method(
        config.mcp_endpoint,
        "notifications/initialized",
        {},
        headers,
        session_id=result.get('session_id')  # Use session ID from earlier test
    )
    print(f"\nInitialized notification sent")

ESTABLISHING MCP SESSION
Session established!
Server: {'name': 'atlassian-mcp-server', 'version': '1.0.0'}

Initialized notification sent


In [12]:
# Probe for available methods - test various MCP method names
# to discover what the Atlassian server actually supports

print("=" * 60)
print("PROBING ATLASSIAN MCP SERVER METHODS")
print("=" * 60)

# Use the session ID from previous initialize
mcp_session_id = result.get('session_id') if 'result' in dir() else None
print(f"Using session ID: {mcp_session_id}")
print()

# Methods to test
methods_to_test = [
    # Standard MCP methods
    ("tools/list", {}),
    ("resources/list", {}),
    ("prompts/list", {}),
    
    # Alternative method names
    ("list_tools", {}),
    ("listTools", {}),
    ("get_tools", {}),
    
    # Atlassian-specific guesses
    ("confluence/search", {"query": "test"}),
    ("jira/search", {"query": "test"}),
    ("search", {"query": "test"}),
    ("query", {"query": "test", "product": "confluence"}),
    
    # Tool call attempts with different tool names
    ("tools/call", {"name": "confluence_search", "arguments": {"query": "test"}}),
    ("tools/call", {"name": "search", "arguments": {"query": "test"}}),
    ("tools/call", {"name": "confluence-search", "arguments": {"query": "test"}}),
    ("tools/call", {"name": "searchConfluence", "arguments": {"query": "test"}}),
]

probe_headers = build_mcp_headers(config, session_id=mcp_session_id)

print("Testing methods...")
print()

for method, params in methods_to_test:
    probe_result = send_mcp_method(
        config.mcp_endpoint,
        method,
        params,
        probe_headers,
        session_id=mcp_session_id,
    )
    
    if probe_result["success"]:
        response = probe_result["response"]
        if "error" in response:
            error_msg = response.get("error", {}).get("message", "Unknown error")
            print(f"  {method}: ERROR - {error_msg}")
        else:
            print(f"  {method}: SUCCESS")
            print(f"    Response: {json.dumps(response.get('result', response))[:200]}")
    else:
        print(f"  {method}: FAILED - {probe_result.get('error', 'Unknown')[:50]}")

PROBING ATLASSIAN MCP SERVER METHODS
Using session ID: r13-060b9c1a-9070-4e7c-a719-a9042444a143

Testing methods...

  tools/list: ERROR - Method not found
  resources/list: ERROR - Method not found
  prompts/list: ERROR - Method not found
  list_tools: ERROR - Method not found
  listTools: ERROR - Method not found
  get_tools: ERROR - Method not found
  confluence/search: ERROR - Method not found
  jira/search: ERROR - Method not found
  search: ERROR - Method not found
  query: ERROR - Method not found
  tools/call: ERROR - Method not found
  tools/call: ERROR - Method not found
  tools/call: ERROR - Method not found
  tools/call: ERROR - Method not found


In [13]:
async def query_confluence_mcp(
    endpoint: str,
    query: str,
    headers: Optional[Dict[str, str]] = None,
    tool_name: str = "confluence_search",
) -> Dict[str, Any]:
    """Query Confluence using the MCP server.
    
    Args:
        endpoint: MCP server endpoint
        query: Search query or question
        headers: Optional auth headers
        tool_name: Name of the search tool to use
    
    Returns:
        Query result with content or error information
        
    Note: Uses httpx.AsyncClient with verify=False to bypass SSL verification
    in corporate proxy environments.
    """
    result = {
        "success": False,
        "query": query,
        "tool_used": None,
        "response": None,
        "error": None,
    }
    
    if not MCP_SDK_AVAILABLE or not STREAMABLE_HTTP_AVAILABLE:
        result["error"] = "MCP SDK not available"
        return result
    
    try:
        custom_headers = headers or {}
        
        # Create httpx.AsyncClient with headers and SSL bypass
        async with httpx.AsyncClient(headers=custom_headers, verify=False) as http_client:
            async with streamable_http_client(endpoint, http_client=http_client) as (read, write, _):
                async with ClientSession(read, write) as session:
                    await session.initialize()
                    
                    # List tools to find the right one
                    tools_result = await session.list_tools()
                    available_tools = {t.name: t for t in tools_result.tools}
                    
                    # Find a search/query tool
                    search_tools = [
                        "confluence_search", "search", "search_confluence",
                        "confluence_query", "query", "find"
                    ]
                    
                    tool_to_use = None
                    for name in search_tools:
                        if name in available_tools:
                            tool_to_use = name
                            break
                    
                    # If no known tool, try the first available
                    if not tool_to_use and available_tools:
                        tool_to_use = list(available_tools.keys())[0]
                    
                    if not tool_to_use:
                        result["error"] = "No search tools available"
                        return result
                    
                    result["tool_used"] = tool_to_use
                    logger.info(f"Using tool: {tool_to_use}")
                    
                    # Call the tool
                    tool_result = await session.call_tool(
                        tool_to_use,
                        arguments={"query": query}
                    )
                    
                    result["success"] = True
                    result["response"] = tool_result
                
    except Exception as e:
        result["error"] = str(e)
        logger.error(f"Query failed: {e}")
    
    return result

print("Query function defined (with SSL bypass and correct httpx.AsyncClient pattern).")

Query function defined (with SSL bypass and correct httpx.AsyncClient pattern).


In [14]:
# Test a simple Confluence query
if MCP_SDK_AVAILABLE and STREAMABLE_HTTP_AVAILABLE:
    test_query = "What projects are documented in Confluence?"
    
    print(f"Query: {test_query}")
    print()
    
    # Build headers
    query_headers = {}
    if config.api_token and config.user_email:
        import base64
        credentials = f"{config.user_email}:{config.api_token}"
        encoded = base64.b64encode(credentials.encode()).decode()
        query_headers["Authorization"] = f"Basic {encoded}"
    
    query_result = await query_confluence_mcp(
        config.mcp_endpoint,
        test_query,
        query_headers
    )
    
    print("=" * 60)
    print("QUERY RESULTS")
    print("=" * 60)
    print(f"Success: {query_result['success']}")
    print(f"Tool Used: {query_result['tool_used']}")
    
    if query_result['response']:
        print("\nResponse:")
        print(str(query_result['response'])[:2000])
    
    if query_result['error']:
        print(f"\nError: {query_result['error']}")
else:
    print("Cannot run query - MCP SDK not available")

Query: What projects are documented in Confluence?



09:52:10 | ERROR    | Query failed: unhandled errors in a TaskGroup (1 sub-exception)


QUERY RESULTS
Success: False
Tool Used: None

Error: unhandled errors in a TaskGroup (1 sub-exception)


## 7. Interactive Query (Optional)

Use this cell to run custom queries against Confluence via the MCP server.

In [13]:
# Set your custom query here
CUSTOM_QUERY = "Find documentation about data pipelines"

if MCP_SDK_AVAILABLE and STREAMABLE_HTTP_AVAILABLE:
    print(f"Running custom query: {CUSTOM_QUERY}")
    print()
    
    custom_result = await query_confluence_mcp(
        config.mcp_endpoint,
        CUSTOM_QUERY,
        query_headers if 'query_headers' in dir() else {}
    )
    
    if custom_result['success']:
        print("Query successful!")
        print("\nResponse:")
        print(str(custom_result['response'])[:3000])
    else:
        print(f"Query failed: {custom_result['error']}")
else:
    print("MCP SDK not available for queries")

Running custom query: Find documentation about data pipelines



09:48:14 | ERROR    | Query failed: unhandled errors in a TaskGroup (1 sub-exception)


Query failed: unhandled errors in a TaskGroup (1 sub-exception)


## 8. Diagnostic Summary

Summary of all tests and recommendations.

In [ ]:
print("=" * 60)
print("ATLASSIAN MCP SERVER TEST SUMMARY")
print("=" * 60)
print(f"Timestamp: {datetime.now().isoformat()}")
print()

# Dependencies
print("Dependencies:")
print(f"  MCP SDK: {'Available' if MCP_SDK_AVAILABLE else 'NOT INSTALLED'}")
print(f"  httpx: {'Available' if HTTPX_AVAILABLE else 'Not installed'}")
print()

# Configuration
print("Configuration:")
print(f"  Site URL: {'Configured' if config.site_url else 'NOT SET'}")
print(f"  API Token: {'Configured' if config.api_token else 'NOT SET (OAuth required)'}")
print()

# Connectivity
print("Connectivity:")
if 'connectivity' in dir():
    print(f"  DNS Resolution: {'OK' if connectivity['dns_resolved'] else 'FAILED'}")
    print(f"  HTTP Reachable: {'OK' if connectivity['reachable'] else 'FAILED'}")
    print(f"  Status Code: {connectivity['status_code']}")
print()

# MCP Protocol
print("MCP Protocol:")
if 'result' in dir():
    print(f"  Initialize: {'OK' if result['success'] else 'FAILED'}")
    print(f"  Session ID: {'Received' if result['session_id'] else 'None'}")
print()

# Recommendations
print("Recommendations:")
recommendations = []

if not MCP_SDK_AVAILABLE:
    recommendations.append("Install MCP SDK: pip install 'mcp[cli]'")

if not config.site_url:
    recommendations.append("Set ATLASSIAN_SITE_URL environment variable")

if not config.api_token:
    recommendations.append("Set ATLASSIAN_API_TOKEN or use OAuth browser flow")

if 'connectivity' in dir() and not connectivity['reachable']:
    recommendations.append("Check network/firewall settings for mcp.atlassian.com")

if 'result' in dir() and result.get('status_code') == 401:
    recommendations.append("Authentication failed - verify credentials or use OAuth")

if recommendations:
    for rec in recommendations:
        print(f"  - {rec}")
else:
    print("  All checks passed!")

print()
print("=" * 60)

## Next Steps

If the connection is successful:
1. Explore the available tools using `session.list_tools()`
2. Test different search queries
3. Integrate MCP queries into the RAG pipeline

If the connection failed:
1. Check the diagnostic summary above
2. Verify your Atlassian site is configured for MCP access
3. Contact your Atlassian admin to enable API token authentication
4. Try the OAuth flow for browser-based authentication

## Resources
- [Atlassian MCP Server Docs](https://support.atlassian.com/atlassian-rovo-mcp-server/)
- [MCP Protocol Specification](https://modelcontextprotocol.io/)
- [MCP Python SDK](https://pypi.org/project/mcp/)